# Reproduce manuscript Figures 10 and 11
Run the next cell only. The input CSVs bundled in `paper_inputs/` come from the CNSPlots/repository saved evaluations. The PhoBERT-large results are computed from the recorded held-out confusion matrices. No model retraining is involved.


In [ ]:
# One self-contained Jupyter cell. No training, API key, or internet needed.
# Reads published model-evaluation CSVs; computes PhoBERT-large
# accuracy and class-1 F1 from its recorded confusion-matrix counts.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Find the source CSVs beside the notebook, in paper_inputs/, or in the
# extracted CNSPlots_Complete_Runnable and vietnamese-depression-nlp folders.
_here = Path.cwd().resolve()
_roots = [
    _here, *_here.parents, _here / 'paper_inputs',
    _here / 'CNSPlots_Complete_Runnable' / 'paper_inputs',
    _here / 'vietnamese-depression-nlp',
    _here.parent, _here.parent / 'paper_inputs',
]

def find_source(rel):
    for root in _roots:
        p = root / rel
        if p.is_file():
            return p
    raise FileNotFoundError(
        f'Missing {rel}. Extract the complete CNSPlots/repository bundle '
        'or place its paper_inputs folder beside this notebook.'
    )

repro_file = find_source('reproduction_results/all_reproduced_results.csv')
gpt_file = find_source('gpt56_sol/results/gpt_final_metrics_summary.csv')
repro = pd.read_csv(repro_file)
gpt = pd.read_csv(gpt_file)
print('Source 1:', repro_file)
print('Source 2:', gpt_file)

ORDER = [
    'TF-IDF + LR', 'TF-IDF + SVM', 'Word2Vec + BiLSTM',
    'Word2Vec + CNN', 'PhoBERT-base', 'PhoBERT-large', 'GPT-5.6 Sol'
]
records = []
for _, r in repro.iterrows():
    records.append((str(r['model']), f"D{int(r['dataset'])}",
                    float(r['accuracy']), float(r['f1_class1'])))
for _, r in gpt.iterrows():
    records.append(('GPT-5.6 Sol', str(r['dataset']),
                    float(r['accuracy']), float(r['f1_class1'])))

# PhoBERT-large's recorded final confusion matrices (not row-level predictions).
# Columns: true negatives, false positives, false negatives, true positives.
PHOBERT_LARGE_COUNTS = {'D1': (422, 8, 5, 752),
                        'D2': (706, 7, 6, 750)}
for ds, (tn, fp, fn, tp) in PHOBERT_LARGE_COUNTS.items():
    accuracy = (tn + tp) / (tn + fp + fn + tp)
    class1_f1 = 2 * tp / (2 * tp + fp + fn)
    records.append(('PhoBERT-large', ds, accuracy, class1_f1))

perf_figure = pd.DataFrame(records, columns=['model', 'dataset', 'accuracy', 'f1_class1'])
assert len(perf_figure) == 14, 'Expected seven models on two datasets.'
assert not perf_figure.duplicated(['model', 'dataset']).any()
assert set(perf_figure['model']) == set(ORDER)
assert set(perf_figure['dataset']) == {'D1', 'D2'}

# Check exact manuscript table values (percentages rounded to two decimals).
CHECK = {
    'TF-IDF + LR':         [(96.62, 97.30), (97.17, 97.25)],
    'TF-IDF + SVM':        [(97.13, 97.72), (97.10, 97.20)],
    'Word2Vec + BiLSTM':   [(96.97, 97.62), (97.21, 97.30)],
    'Word2Vec + CNN':      [(97.89, 98.36), (97.96, 98.03)],
    'PhoBERT-base':        [(97.56, 98.10), (98.09, 98.15)],
    'PhoBERT-large':       [(98.90, 99.14), (99.12, 99.14)],
    'GPT-5.6 Sol':         [(93.77, 94.90), (93.53, 93.40)],
}
for model in ORDER:
    for ds, expected in zip(('D1', 'D2'), CHECK[model]):
        r = perf_figure.query('model == @model and dataset == @ds').iloc[0]
        actual = (round(100 * r['accuracy'], 2), round(100 * r['f1_class1'], 2))
        if actual != expected:
            raise ValueError(f'Manuscript mismatch: {model} {ds}: {actual} != {expected}')
print('Verified: 14 model–dataset records match manuscript Table 3.')

# Same visual language as the master notebook; PDF only.
BLUE, RED = '#5189BB', '#D6372E'
plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['DejaVu Serif', 'Liberation Serif'],
    'font.size': 9.5, 'axes.labelsize': 10,
    'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'legend.fontsize': 8, 'pdf.fonttype': 42,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white'
})
# If running inside the master plotting notebook, overwrite its PDF outputs.
_figure_root = next((p for p in [_here, *_here.parents] if (p/'src').is_dir() and (p/'reproduction_results').is_dir()), _here)
_master_pdf_dir = _figure_root / 'paper_materials_cnsplots' / 'figures' / 'pdf'
out_dir = _master_pdf_dir if _master_pdf_dir.is_dir() else (_figure_root / 'figures')
out_dir.mkdir(parents=True, exist_ok=True)

def draw_pair(metric, label, name):
    fig, ax = plt.subplots(figsize=(6.7, 4.0))
    ypos = np.arange(len(ORDER))[::-1]
    for y, model in zip(ypos, ORDER):
        v = perf_figure[perf_figure.model == model].set_index('dataset')
        d1, d2 = 100 * v.loc['D1', metric], 100 * v.loc['D2', metric]
        ax.plot([d1, d2], [y, y], color='#BDBDBD', lw=0.8, zorder=1)
        for x, col, vertical, valign in (
            (d1, BLUE, +0.20, 'bottom'),
            (d2, RED, -0.20, 'top'),
        ):
            ax.scatter(x, y, s=30, color=col, edgecolor='white', lw=0.45, zorder=3)
            ax.text(x, y + vertical, f'{x:.2f}', color=col,
                    ha='center', va=valign, fontsize=7.8)
    ax.set_yticks(ypos, ORDER)
    ax.set_xlim(92.5, 100)
    ax.set_ylim(-0.6, 6.65)
    ax.set_xlabel(label)
    ax.set_axisbelow(True)
    ax.grid(axis='x', color='#D9DDE2', alpha=0.65, linewidth=0.55)
    ax.tick_params(length=3, width=0.75)
    ax.legend(handles=[
        Line2D([], [], color=BLUE, marker='o', linestyle='none', label='D1'),
        Line2D([], [], color=RED, marker='o', linestyle='none', label='D2'),
    ], loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=2,
       frameon=True, fancybox=False)
    fig.subplots_adjust(left=0.26, right=0.99, top=0.97, bottom=0.21)
    path = out_dir / name
    fig.savefig(path, format='pdf', bbox_inches='tight', pad_inches=0.06)
    plt.show()
    plt.close(fig)
    print('Saved:', path)

draw_pair('accuracy', 'Accuracy (%)', 'benchmark_model_accuracy.pdf')
draw_pair('f1_class1', 'F1 for depression-related-signal class (%)',
          'benchmark_model_class1_f1.pdf')
